In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings("ignore")

# =========================
# CONFIG
# =========================
ROOT = Path("/home/mohamed/SDD/hackathon/sia-predicting-short-form-video-popularity")
N_SPLITS = 5
SEED = 42

# =========================
# 1) Chargement
# =========================
train = pd.read_csv(ROOT / "train_full_merged.csv")
test  = pd.read_csv(ROOT / "test_full_merged.csv")
y_df  = pd.read_csv(ROOT / "y_train.csv",sep=";")

# Harmoniser ID
def normalize_id(s):
    s = s.astype(str)
    for pat in [r"^VIDEO_", r"^video_", r"^Video_"]:
        s = s.str.replace(pat, "", regex=True)
    return s

train["ID"] = normalize_id(train["ID"])
test["ID"]  = normalize_id(test["ID"])
y_df["ID"]  = normalize_id(y_df["ID"] if "ID" in y_df.columns else y_df.iloc[:, 0])

# Merge target
train = train.merge(y_df[["ID", "popularity"]], on="ID", how="left")
print(f"Train shape: {train.shape} | Target nulls: {train['popularity'].isna().sum()}")
print(f"Test shape : {test.shape}")

# =========================
# 2) Préparer features
# =========================
drop_cols = ["ID", "popularity"]

# Colonnes non numériques → drop (LightGBM veut du numérique ou category)
non_numeric = train.drop(columns=drop_cols, errors="ignore").select_dtypes(exclude=[np.number]).columns.tolist()
print(f"\nColonnes non-numériques droppées ({len(non_numeric)}): {non_numeric[:10]}{'...' if len(non_numeric)>10 else ''}")

feature_cols = [c for c in train.columns if c not in drop_cols + non_numeric]

X = train[feature_cols].copy()
y = train["popularity"].copy()
X_test_final = test[feature_cols].copy()

print(f"\nNombre de features: {len(feature_cols)}")
print(f"NaN dans X: {X.isna().sum().sum()} | NaN dans X_test: {X_test_final.isna().sum().sum()}")

# =========================
# 3) LightGBM params
# =========================
lgb_params = {
    "objective":        "regression",
    "metric":           "rmse",
    "learning_rate":    0.03,
    "num_leaves":       127,
    "max_depth":        -1,
    "min_child_samples": 20,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq":     5,
    "reg_alpha":        0.1,
    "reg_lambda":       1.0,
    "n_jobs":           -1,
    "verbose":          -1,
    "random_state":     SEED,
}

# =========================
# 4) Cross-validation + OOF
# =========================
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

oof_preds   = np.zeros(len(X))
test_preds  = np.zeros(len(X_test_final))
feature_imp = np.zeros(len(feature_cols))

print(f"\n{'='*50}")
print(f"KFold CV — {N_SPLITS} folds")
print(f"{'='*50}")

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = lgb.LGBMRegressor(n_estimators=3000, **lgb_params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[
            lgb.early_stopping(stopping_rounds=100, verbose=False),
            lgb.log_evaluation(period=200)
        ]
    )

    oof_preds[val_idx] = model.predict(X_val)
    test_preds         += model.predict(X_test_final) / N_SPLITS
    feature_imp        += model.feature_importances_ / N_SPLITS

    fold_rmse = np.sqrt(mean_squared_error(y_val, oof_preds[val_idx]))
    print(f"  Fold {fold+1} | Best iter: {model.best_iteration_:4d} | RMSE: {fold_rmse:.4f}")

oof_rmse = np.sqrt(mean_squared_error(y, oof_preds))
print(f"\n{'='*50}")
print(f"OOF RMSE global : {oof_rmse:.4f}")
print(f"{'='*50}")

# =========================
# 5) Feature importance top 20
# =========================
fi_df = pd.DataFrame({"feature": feature_cols, "importance": feature_imp})
fi_df = fi_df.sort_values("importance", ascending=False).head(20)
print("\nTop 20 features:")
print(fi_df.to_string(index=False))

# =========================
# 6) Submission
# =========================
submission = pd.DataFrame({
    "ID":         test["ID"].values,
    "popularity": test_preds
})

# Vérif
assert submission["ID"].isna().sum() == 0, "IDs manquants dans la submission!"
assert submission["popularity"].isna().sum() == 0, "Prédictions NaN dans la submission!"

out_path = ROOT / "submission_lgbm.csv"
submission.to_csv(out_path, index=False)

print(f"\n✅ Submission sauvegardée : {out_path}")
print(f"   Shape : {submission.shape}")
print(f"\nAperçu:")
print(submission.head(10).to_string(index=False))
print(f"\nStats popularity prédit:")
print(submission["popularity"].describe().round(4))

Train shape: (1348, 105) | Target nulls: 0
Test shape : (338, 104)

Colonnes non-numériques droppées (5): ['album', 'artist', 'channel', 'track', 'uploader']

Nombre de features: 98
NaN dans X: 66 | NaN dans X_test: 9

KFold CV — 5 folds
[200]	valid_0's rmse: 1.24724
  Fold 1 | Best iter:  231 | RMSE: 1.2424
[200]	valid_0's rmse: 1.22954
  Fold 2 | Best iter:  125 | RMSE: 1.2273
[200]	valid_0's rmse: 1.23896
  Fold 3 | Best iter:  140 | RMSE: 1.2313
[200]	valid_0's rmse: 1.34533
  Fold 4 | Best iter:  130 | RMSE: 1.3438
[200]	valid_0's rmse: 1.32653
  Fold 5 | Best iter:  110 | RMSE: 1.3195

OOF RMSE global : 1.2737

Top 20 features:
          feature  importance
    uploader_freq        96.4
lum_f3_brightness        95.8
lum_f1_brightness        91.4
    download_hour        87.6
     aud_rms_mean        85.8
  aud_mfcc_3_mean        85.6
 lum_f1_sharpness        84.8
   dyn_avg_motion        83.2
lum_f3_saturation        83.0
      aud_rms_std        81.6
           col_W2        79.